# SurfaceTarget-CRC — phase 01 — patch 1 8

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.8
#
# Published TCGA MSI Classification Retrieval
#
# Source:
# cBioPortal - coadread_tcga_pub
#
# Objectives:
# 1. Retrieve published MSI_STATUS annotations
# 2. Inspect actual categorical values
# 3. Identify patient-level annotation conflicts
# 4. Match annotations to the GDC RNA-seq cohort
# 5. Compare with preliminary computational MSI groups
#
# IMPORTANT:
# No final MSS cohort assignment is performed.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import requests
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

RESULTS_DIR = PROJECT_ROOT / "results" / "tables"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CBIO_API = "https://www.cbioportal.org/api"

STUDY_ID = "coadread_tcga_pub"

ATTRIBUTE_ID = "MSI_STATUS"

COHORT_PATH = (
    METADATA_DIR /
    "tcga_msi_preliminary_patient_classification.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.8 — PUBLISHED MSI CLASSIFICATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load preliminary cohort
# ------------------------------------------------------------

cohort = pd.read_csv(
    COHORT_PATH,
    dtype={"patientId": str}
)

if cohort["patientId"].duplicated().any():
    raise RuntimeError(
        "Duplicate patients in preliminary cohort."
    )

print("\nPreliminary cohort loaded.")

print("Total patients:", len(cohort))

print("\nExisting MSI groups:")

print(
    cohort["msi_group"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 3. Initialize API session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})

# ------------------------------------------------------------
# 4. Verify published MSI attribute
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PUBLISHED MSI ATTRIBUTE VERIFICATION")
print("-" * 65)

attributes_url = (
    f"{CBIO_API}/studies/"
    f"{STUDY_ID}/clinical-attributes"
)

response = session.get(
    attributes_url,
    params={"pageSize": 10000},
    timeout=60
)

response.raise_for_status()

attributes = response.json()

matches = [
    a for a in attributes
    if a.get("clinicalAttributeId") == ATTRIBUTE_ID
]

if len(matches) != 1:
    raise RuntimeError(
        "MSI_STATUS attribute could not be uniquely verified."
    )

attribute = matches[0]

print("\nStudy:", STUDY_ID)

print("Attribute ID:", ATTRIBUTE_ID)

print(
    "Display name:",
    attribute.get("displayName")
)

print(
    "Data type:",
    attribute.get(
        "datatype",
        attribute.get("dataType", "Not specified")
    )
)

print(
    "Patient-level:",
    attribute.get("patientAttribute")
)

if attribute.get("patientAttribute") is True:
    raise RuntimeError(
        "MSI_STATUS is patient-level; "
        "the retrieval method needs adjustment."
    )

print("\nAttribute verification: PASSED")

# ------------------------------------------------------------
# 5. Retrieve published MSI classifications
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RETRIEVING PUBLISHED MSI STATUS")
print("-" * 65)

endpoint = (
    f"{CBIO_API}/studies/"
    f"{STUDY_ID}/clinical-data"
)

all_records = []

PAGE_SIZE = 500
page_number = 0

while True:

    params = {
        "clinicalDataType": "SAMPLE",
        "attributeId": ATTRIBUTE_ID,
        "pageSize": PAGE_SIZE,
        "pageNumber": page_number
    }

    response = session.get(
        endpoint,
        params=params,
        timeout=90
    )

    response.raise_for_status()

    records = response.json()

    if not isinstance(records, list):
        raise RuntimeError(
            "Unexpected clinical API response."
        )

    # Ensure the API returned only MSI_STATUS.

    unexpected = {
        record.get("clinicalAttributeId")
        for record in records
        if record.get("clinicalAttributeId") != ATTRIBUTE_ID
    }

    if unexpected:
        raise RuntimeError(
            f"Unexpected attributes returned: {unexpected}"
        )

    all_records.extend(records)

    print(
        f"Page {page_number + 1}: "
        f"{len(records)} records"
    )

    if len(records) < PAGE_SIZE:
        break

    page_number += 1

    if page_number > 100:
        raise RuntimeError(
            "Pagination safety limit exceeded."
        )

if not all_records:
    raise RuntimeError(
        "No published MSI_STATUS records retrieved."
    )

# ------------------------------------------------------------
# 6. Create published MSI dataframe
# ------------------------------------------------------------

published_msi = pd.DataFrame(all_records)

required_columns = [
    "patientId",
    "sampleId",
    "clinicalAttributeId",
    "value"
]

missing = [
    col for col in required_columns
    if col not in published_msi.columns
]

if missing:
    raise RuntimeError(
        f"Required columns missing: {missing}"
    )

published_msi = (
    published_msi
    .drop_duplicates()
    .copy()
)

# Preserve original classifications.

published_msi["original_status"] = (
    published_msi["value"]
    .astype("string")
)

# Normalize formatting only.
# Do not combine MSS, MSI-L, or MSI-H.

published_msi["status_clean"] = (
    published_msi["original_status"]
    .str.strip()
    .str.upper()
)

published_msi.loc[
    published_msi["status_clean"].isin(
        ["", "NA", "N/A", "UNKNOWN", "NOT AVAILABLE"]
    ),
    "status_clean"
] = pd.NA

print("\n" + "-" * 65)
print("PUBLISHED MSI DATA SUMMARY")
print("-" * 65)

print(
    "\nTotal records:",
    len(published_msi)
)

print(
    "Unique patients:",
    published_msi["patientId"].nunique()
)

print(
    "Unique samples:",
    published_msi["sampleId"].nunique()
)

# ------------------------------------------------------------
# 7. Examine actual MSI classifications
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PUBLISHED MSI STATUS DISTRIBUTION")
print("-" * 65)

print("\nOriginal classification values:")

print(
    published_msi["original_status"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nNormalized classification values:")

print(
    published_msi["status_clean"]
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------
# 8. Audit multiple annotations per patient
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PATIENT-LEVEL ANNOTATION AUDIT")
print("-" * 65)

patient_summary = (
    published_msi.groupby("patientId")
    .agg(
        n_samples=("sampleId", "nunique"),
        n_unique_statuses=("status_clean", "nunique"),
        published_status=("status_clean", "first")
    )
    .reset_index()
)

conflicts = patient_summary[
    patient_summary["n_unique_statuses"] > 1
].copy()

print(
    "\nPatients with multiple annotated samples:",
    (patient_summary["n_samples"] > 1).sum()
)

print(
    "Patients with conflicting MSI classifications:",
    len(conflicts)
)

if not conflicts.empty:

    print("\nConflicting patient examples:")

    print(
        conflicts.head(10).to_string(index=False)
    )

# Do not assign a single patient-level status
# when annotations are missing or conflicting.

ambiguous = (
    patient_summary["n_unique_statuses"] != 1
)

patient_summary.loc[
    ambiguous,
    "published_status"
] = pd.NA

# ------------------------------------------------------------
# 9. Match published annotations to GDC cohort
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GDC / PUBLISHED MSI PATIENT MATCHING")
print("-" * 65)

comparison = cohort.merge(
    patient_summary,
    on="patientId",
    how="left",
    validate="one_to_one"
)

has_published_status = (
    comparison["published_status"].notna()
)

print("\nGDC primary tumor RNA-seq patients:", len(cohort))

print(
    "Patients with unambiguous published MSI status:",
    has_published_status.sum()
)

print(
    "Patients without usable published MSI status:",
    (~has_published_status).sum()
)

# ------------------------------------------------------------
# 10. Check sample identifier correspondence
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-STUDY SAMPLE ID CHECK")
print("-" * 65)

# Compare published study sample IDs with the
# PanCancer Atlas MSI sample identifiers.

published_sample_sets = (
    published_msi.groupby("patientId")["sampleId"]
    .agg(lambda x: set(x.dropna()))
    .to_dict()
)

sample_check = comparison[
    comparison["published_status"].notna()
    & comparison["mantis_sample_id"].notna()
].copy()

sample_check["exact_sample_match"] = (
    sample_check.apply(
        lambda row:
        str(row["mantis_sample_id"])
        in published_sample_sets.get(
            row["patientId"], set()
        ),
        axis=1
    )
)

print(
    "\nPatients eligible for sample ID comparison:",
    len(sample_check)
)

print(
    "Exact cross-study sample ID matches:",
    sample_check["exact_sample_match"].sum()
)

print(
    "Non-matching sample IDs:",
    (~sample_check["exact_sample_match"]).sum()
)

if (~sample_check["exact_sample_match"]).any():

    print("\nExamples requiring sample-level review:")

    examples = sample_check.loc[
        ~sample_check["exact_sample_match"],
        ["patientId", "mantis_sample_id"]
    ].head(5)

    print(examples.to_string(index=False))

print(
    "\nNOTE: Sample IDs may differ in barcode detail "
    "between studies. Non-matches require review."
)

# ------------------------------------------------------------
# 11. Compare published and computational groups
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PUBLISHED VS COMPUTATIONAL MSI")
print("-" * 65)

available = comparison[
    comparison["published_status"].notna()
].copy()

if available.empty:

    print("\nNo overlapping classifications available.")

else:

    cross_tab = pd.crosstab(
        available["msi_group"],
        available["published_status"]
    )

    print("\nClassification comparison:")

    print(cross_tab.to_string())

# ------------------------------------------------------------
# 12. Specifically examine preliminary low-MSI cases
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRELIMINARY LOW-MSI COHORT EVALUATION")
print("-" * 65)

low_msi = comparison[
    comparison["msi_group"] == "Concordant low MSI"
].copy()

print(
    "\nTotal preliminary low-MSI patients:",
    len(low_msi)
)

print(
    "With published MSI classification:",
    low_msi["published_status"].notna().sum()
)

print("\nPublished status distribution:")

print(
    low_msi["published_status"]
    .fillna("NOT AVAILABLE")
    .value_counts()
    .to_string()
)

print("\nBy TCGA project:")

project_comparison = pd.crosstab(
    low_msi["gdc_project"],
    low_msi["published_status"].fillna("NOT AVAILABLE")
)

print(project_comparison.to_string())

# ------------------------------------------------------------
# 13. Save research outputs
# ------------------------------------------------------------

raw_path = (
    METADATA_DIR /
    "tcga_published_msi_status_raw.csv"
)

patient_path = (
    METADATA_DIR /
    "tcga_published_msi_patient_summary.csv"
)

comparison_path = (
    METADATA_DIR /
    "tcga_computational_vs_published_msi.csv"
)

published_msi.to_csv(
    raw_path,
    index=False
)

patient_summary.to_csv(
    patient_path,
    index=False
)

comparison.to_csv(
    comparison_path,
    index=False
)

# ------------------------------------------------------------
# 14. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "cBioPortal",

    "study_id": STUDY_ID,

    "attribute": ATTRIBUTE_ID,

    "total_records": int(len(published_msi)),

    "unique_patients": int(
        published_msi["patientId"].nunique()
    ),

    "gdc_overlap_with_usable_status": int(
        has_published_status.sum()
    ),

    "patients_with_conflicting_annotations": int(
        len(conflicts)
    ),

    "final_mss_cohort_assigned": False,

    "limitations": [
        "Published classifications may cover only a subset of TCGA patients.",
        "Cross-study sample identifiers require verification.",
        "Published MSI and computational scores may not be independent.",
        "No final MSS cohort has been established."
    ]
}

provenance_path = (
    METADATA_DIR /
    "tcga_published_msi_provenance.json"
)

with open(provenance_path, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 15. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.8 SUMMARY")
print("=" * 65)

print("\nFiles saved:")

print("1. tcga_published_msi_status_raw.csv")

print("2. tcga_published_msi_patient_summary.csv")

print("3. tcga_computational_vs_published_msi.csv")

print("4. tcga_published_msi_provenance.json")

print("\nNo final MSS cohort assigned.")

print("No molecular expression datasets downloaded.")

print("\nPATCH 1.8 COMPLETE")